# Auspify Technologies Data Science Internship
## Task 5: Machine Learning Classification Model (Netflix Dataset)

**Intern:** Rishi Prajapati  
**Date:** November 06, 2026  
**Objective:** Build, evaluate, and benchmark multi-class Machine Learning classification models to automatically predict the target demographic tier of Netflix titles based on content genres, title text, format, origin country, and release era.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add custom modules to sys.path
sys.path.append('../src')
from data_preparer import load_and_filter_data, prepare_train_test_split
from model_trainer import train_and_compare_models
from evaluator import evaluate_classifier_performance
from visualizer import plot_algorithm_comparison, plot_confusion_matrix_heatmap

print('Machine learning classification modules loaded successfully!')

### Step 1: Ingestion & Demographic Class Distribution
Load the dataset and analyze demographic categories: Adults (TV-MA, R), Teens (TV-14, PG-13), Older Kids (TV-PG, PG), and Kids (TV-Y, G).

In [2]:
df = load_and_filter_data('../data/netflix_cleaned.csv')
X_train, X_test, y_train, y_test = prepare_train_test_split(df)

plt.figure(figsize=(8, 4))
df['target_audience'].value_counts().plot(kind='bar', color=['#E50914', '#221F1F', '#4A5568', '#CBD5E0'], edgecolor='black')
plt.title('Target Audience Class Distribution in Netflix Catalog')
plt.ylabel('Title Count')
plt.xticks(rotation=0)
plt.show()

### Step 2: Multi-Algorithm Model Training & Benchmark
Train Logistic Regression, Linear SVM, Multinomial Naive Bayes, and Random Forest Classifier.

In [3]:
comparison_df, pipelines, best_algo = train_and_compare_models(X_train, X_test, y_train, y_test)
comparison_df

### Step 3: Granular Classification Evaluation & Confusion Matrix
Examine per-class precision, recall, and normalized confusion matrix for the winning model.

In [4]:
best_pipe = pipelines[best_algo]
report_dict, cm, labels = evaluate_classifier_performance(best_pipe, X_test, y_test)

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(7, 5.5))
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Reds', xticklabels=labels, yticklabels=labels, linewidths=1, linecolor='black')
plt.title(f'Normalized Confusion Matrix - {best_algo}')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()

### Step 4: Sample Live Predictions
Test pipeline inference on new title profiles.

In [5]:
sample_input = pd.DataFrame([
    {
        'listed_in': 'Children & Family Movies, Comedies',
        'title': 'The Magical Animal Forest',
        'type': 'Movie',
        'primary_country': 'United States',
        'release_year': 2021
    },
    {
        'listed_in': 'Crime TV Shows, TV Dramas, Action',
        'title': 'Underworld Cartel',
        'type': 'TV Show',
        'primary_country': 'Mexico',
        'release_year': 2018
    }
])

preds = best_pipe.predict(sample_input)
for title, pred in zip(sample_input['title'], preds):
    print(f'Title: "{title}" -> Predicted Audience Tier: {pred}')